In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [12]:
# Load the CSV file
df = pd.read_csv(r"C:\DTT ML 2026\06-trees\car_fuel_efficiency_2026.csv")

In [13]:
# Check that the dataset loaded properly and view the first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# 1. Fill missing values with zeros
df = df.fillna(0)

# 2. Split with the exact specified random_state and ratios
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 3. Extract target variable
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove target from feature matrices
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# 4. Vectorize features using DictVectorizer
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

feature_names = dv.get_feature_names_out()
print(f"Training matrix shape: {X_train.shape}")
print(f"Validation matrix shape: {X_val.shape}")

Training matrix shape: (6000, 16)
Validation matrix shape: (2000, 16)


In [15]:
# Q1
from sklearn.tree import DecisionTreeRegressor, export_text

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(feature_names)))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



The decision tree split at the root node is on model_year (threshold 1997.50).

Question 1 Answer: model_year

In [16]:
# Q2
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"Validation RMSE: {rmse_val:.4f}")

Validation RMSE: 1.8332


The difference between your result (1.8332) and the option (1.837) is approximately 0.0038

RMSE of 1.8332 corresponds to the second order-of-magnitude option.

Question 2 Answer: 1.837

In [20]:
# Q3
scores_q3 = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores_q3[n] = rmse
    print(f"n_estimators={n:3d} | RMSE: {rmse:.4f} | 3 decimals: {rmse:.3f}")

best_n = min(scores_q3, key=scores_q3.get)
print(f"\nLowest validation RMSE: n_estimators = {best_n}")

n_estimators= 10 | RMSE: 1.8332 | 3 decimals: 1.833
n_estimators= 50 | RMSE: 1.7740 | 3 decimals: 1.774
n_estimators=100 | RMSE: 1.7698 | 3 decimals: 1.770
n_estimators=150 | RMSE: 1.7692 | 3 decimals: 1.769

Lowest validation RMSE: n_estimators = 150


Question 3 Answer: 150 (lowest validation RMSE at 1.769).

In [21]:
# Q4
depths = [10, 15, 20, 25]
mean_depth_scores = {}

for d in depths:
    depth_rmses = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n, 
            max_depth=d, 
            random_state=1, 
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        depth_rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    
    mean_rmse = np.mean(depth_rmses)
    mean_depth_scores[d] = mean_rmse
    print(f"max_depth={d:2d} | Mean RMSE: {mean_rmse:.4f}")

best_depth = min(mean_depth_scores, key=mean_depth_scores.get)
print(f"\nBest max_depth (lowest mean RMSE): {best_depth}")

max_depth=10 | Mean RMSE: 1.7544
max_depth=15 | Mean RMSE: 1.7842
max_depth=20 | Mean RMSE: 1.7868
max_depth=25 | Mean RMSE: 1.7863

Best max_depth (lowest mean RMSE): 10


Question 4 Answer: 10 (lowest mean RMSE at 1.7544).

In [22]:
# Q5
rf_q5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_q5.fit(X_train, y_train)

df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_q5.feature_importances_
}).sort_values('importance', ascending=False)

candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print(df_importance[df_importance['feature'].isin(candidates)])

                feature  importance
15       vehicle_weight    0.203313
8            horsepower    0.078271
4   engine_displacement    0.061281
0          acceleration    0.054998


Question 5 Answer: vehicle_weight (importance score of 0.2033, the highest among the candidates).

In [25]:
# Q6
%pip install -q xgboost

import xgboost as xgb
import re

# 1. Clean feature names (XGBoost rejects characters like '[', ']', '<', '=')
clean_features = [re.sub(r'[\[\]<>=, ]', '_', col) for col in feature_names]

# 2. Create DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

# 3. Train with eta = 0.3
xgb_params_03 = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_03 = {}
model_03 = xgb.train(
    xgb_params_03,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    evals_result=evals_03,
    verbose_eval=False
)

# 4. Train with eta = 0.1
xgb_params_01 = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_01 = {}
model_01 = xgb.train(
    xgb_params_01,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    evals_result=evals_01,
    verbose_eval=False
)

# 5. Extract final round validation RMSE
rmse_03 = evals_03['val']['rmse'][-1]
rmse_01 = evals_01['val']['rmse'][-1]

print(f"Validation RMSE (eta=0.3 at round 100): {rmse_03:.4f}")
print(f"Validation RMSE (eta=0.1 at round 100): {rmse_01:.4f}")

if rmse_03 < rmse_01:
    print("\nBest eta: 0.3")
elif rmse_01 < rmse_03:
    print("\nBest eta: 0.1")
else:
    print("\nBoth give equal value")

Note: you may need to restart the kernel to use updated packages.
Validation RMSE (eta=0.3 at round 100): 1.8265
Validation RMSE (eta=0.1 at round 100): 1.7248

Best eta: 0.1



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Question 6 Answer: 0.1 (validation RMSE of 1.7248 vs 1.8265 for 0.3).